# 5단계 추가: 공식 평가셋으로 평가 (논문에 없는 추가)
모델 카드에 링크된 `android_benchmark.xlsx` (논문 표 1 의 평가셋, 210개) 로 우리 모델과 공개 모델을 평가한다.
변환·점검 코드는 `src/official_bench.py`, 생성·채점은 5단계와 같은 `src/evaluate.py`. 이 노트북은 **실행만** 한다.

- 런타임: **A100 GPU**. 🔑 Secrets에 `HF_TOKEN` 등록
- 210개 = 함수 20개 × 10 + `irrelevant_function` 10 (negative 4.8%). 질의에 정답 형식이 대부분 들어 있다 (`17:15`, `+913322667890`, `'Nap Time'`)
- 결과는 Drive 의 `MyDrive/octopus_v2/runs/eval_official/` 에 저장 (변환한 `official_test.jsonl` 도 같이 복사 → 랩탑에서 다시 채점 가능)
- ⚠️ 논문 숫자(99.524% 등)는 nexa 가 사람 눈으로 T/F 를 매긴 관대한 기준이다. 같은 출력을 우리 채점기로 매기면
  Octopus-0 은 92.857% (6-1 에서 확인). 그래서 비교는 **같은 채점기로 매긴 숫자끼리** 한다

## 준비 (그냥 실행)

In [1]:
# 5단계 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
# 처음 설치한 뒤에는 '런타임 > 세션 다시 시작' 후 이 칸부터 다시 실행
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
Cloning into '/content/octopus_v2_reproduce'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 188 (delta 100), reused 119 (delta 40), pack-reused 0 

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"   # 4단계에서 학습한 모델
EVAL = "/content/drive/MyDrive/octopus_v2/runs/eval_official"
TEST = "data/raw/official/official_test.jsonl"
os.makedirs(EVAL, exist_ok=True)

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
transformers 4.56.2


---
## 6-0. 엑셀 내려받기 → `official_test.jsonl`
`[우리 RULES 를 어기는 공식 정답]` 1개 (`'+12345678900,555'`, 내선번호) 는 정답을 고치지 않고 그대로 둔다.

In [3]:
!python -m src.official_bench convert
!cp {TEST} {EVAL}/official_test.jsonl

[다운로드] https://huggingface.co/NexaAIDev/Octopus-v2/resolve/main/android_benchmark.xlsx
[저장] data/raw/official/official_test.jsonl: 210개 {'positive': 200, 'negative': 10}, seen_in_train 0개, 질의 중복 6개
[우리 RULES 를 어기는 공식 정답] 1개
   192 Make a phone call to '+12345678900,555' where '555' is the e ["phone_number: bad value '+12345678900,555'"]


---
## 6-1. 채점 기준 점검 (GPU 필요 없음)
엑셀에 nexa 가 기록한 Octopus-0~3 출력을 우리 채점기로 다시 매겨 nexa 의 T/F 와 비교한다.
판정이 다른 줄이 출력된다 (nexa=T 인데 우리는 오답인 경우가 대부분: 따옴표 깨짐, `72.0`, 검색어 범위 등).

In [4]:
!python -m src.official_bench check


== Octopus-0: nexa 판정 209/210 (99.524%) | 우리 엄격 195/210 (92.857%) | 우리 느슨 195/210 (92.857%) | 판정 다름 14개
   nexa=T 우리=args:valid_but_different
     Q Schedule a team meeting for project kickoff with'start_time at 2023-10
     정답 <nexa_12>('Project Kickoff Meeting', '2023-10-01-09-00', '2023-10-01-10-00')<nex
     출력 <nexa_12>('Project Kickoff', '2023-10-01-09-00', '2023-10-01-10-00')<nexa_end>
   nexa=T 우리=args:valid_but_different
     Q How do I get '80s Smash Hits' to play on my Google Nest Hub from Amazo
     정답 <nexa_17>('Amazon Music', "'80s Smash Hits")<nexa_end>
     출력 <nexa_17>('Amazon Music', '80s Smash Hits')<nexa_end>
   nexa=T 우리=args:valid_but_different
     Q Is it possible to stream live jazz music from Tidal on my Google Nest 
     정답 <nexa_17>('Tidal', 'live jazz')<nexa_end>
     출력 <nexa_17>('Tidal', 'live jazz music')<nexa_end>
   nexa=T 우리=args:valid_but_different
     Q I think I forgot to lock the back door, can you check and lock it if i
     정답 <nexa_19>('lock'

---
## 6-2. 우리 모델로 210개 생성
**예측**: 엄격 정확도는? 온도(`72` vs 우리 `72.0`) 10개는 엄격 채점에서 틀릴 것. negative 10개는?
> (여기에 예측)

In [5]:
!python -m src.evaluate generate --model {RUN_DIR} --test-file {TEST} --out {EVAL}/ours/preds.jsonl

[생성] 210개 중 이미 0개 있음 → 210개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval_official/ours/preds.jsonl
I0000 00:00:1791364414.183072    1460 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791364414.266300    1460 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791364416.667800    1460 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:17913644

In [6]:
!python -m src.evaluate score --preds {EVAL}/ours/preds.jsonl --test-file {TEST}

                                                      전체   seen_in_train=false
개수                                                   210                   210
형식 OK                                            98.57%                98.57% 
함수 선택 정확도                                        89.52%                89.52% 
인자 정확도 (함수 맞힌 positive 중)                        75.84%                75.84% 
전체 정확도 ★                                         69.05%                69.05% 
  positive                                       67.50%                67.50% 
  negative                                      100.00%               100.00% 
느슨한 정확도 (논문에 없는 추가)                              73.81%                73.81% 
호출 문자열 완전 일치                                     69.05%                69.05% 

[결과 종류별 개수]
  args:count_mismatch                                    8                     8
  args:rule_violation                                    2                     2
  args:valid_but_different         

---
## 6-3. 공개 모델 `NexaAIDev/Octopus-v2` 로 210개 생성
엑셀의 Octopus-0 (1K per API, full) 기록과 비슷하게 나오면 우리 생성 코드가 nexa 환경을 재현한다는 확인도 된다.
(공개 모델이 Octopus-0~3 중 어느 것인지는 모델 카드에 없다 → 추정으로만 비교)

In [7]:
!python -m src.evaluate generate --model NexaAIDev/Octopus-v2 --test-file {TEST} --out {EVAL}/public/preds.jsonl

[생성] 210개 중 이미 0개 있음 → 210개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval_official/public/preds.jsonl
tokenizer_config.json: 6.35kB [00:00, 4.69MB/s]
tokenizer.model: 100% 4.24M/4.24M [00:00<00:00, 7.36MB/s]
tokenizer.json: 100% 17.5M/17.5M [00:00<00:00, 32.7MB/s]
added_tokens.json: 100% 500/500 [00:00<00:00, 3.74MB/s]
special_tokens_map.json: 100% 957/957 [00:00<00:00, 10.2MB/s]
config.json: 100% 673/673 [00:00<00:00, 5.47MB/s]
I0000 00:00:1791364586.886010    2290 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791364586.958954    2290 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with 

In [8]:
!python -m src.evaluate score --preds {EVAL}/public/preds.jsonl --test-file {TEST}

                                                      전체   seen_in_train=false
개수                                                   210                   210
형식 OK                                            97.14%                97.14% 
함수 선택 정확도                                        97.14%                97.14% 
인자 정확도 (함수 맞힌 positive 중)                        95.36%                95.36% 
전체 정확도 ★                                         92.86%                92.86% 
  positive                                       92.50%                92.50% 
  negative                                      100.00%               100.00% 
느슨한 정확도 (논문에 없는 추가)                              92.86%                92.86% 
호출 문자열 완전 일치                                     92.86%                92.86% 

[결과 종류별 개수]
  args:valid_but_different                               9                     9
  correct                                              195                   195
  format:unparsable                

---
## 6-4. 나란히 보기
- `ours`, `public`: 이 노트북에서 생성 → 우리 채점기
- `nexa_O0` ~ `nexa_O3`: 엑셀에 기록된 Octopus-0~3 출력 → 우리 채점기 (지연 시간 없음)
- 맨 아래 `nexa 판정` 은 엑셀의 T/F 비율 (= 논문 표 1 숫자)

In [9]:
import json
from src.evaluate import METRIC_NAMES, judge, summarize
from src.official_bench import OCTOPUS_COLUMNS, load_sheet
from src.preprocess.prompt import load_jsonl
from pathlib import Path

test = load_jsonl(Path(TEST))
sheet = load_sheet()
cols = {n: json.load(open(f"{EVAL}/{n}/summary.json", encoding="utf-8"))["전체"] for n in ("ours", "public")}
nexa_tf = {}
for i, name in enumerate(OCTOPUS_COLUMNS.values()):
    scored = [{**row, **judge(row, s[name]["output"] or "")} for s, row in zip(sheet, test)]
    cols[f"nexa_O{i}"] = summarize(scored)
    nexa_tf[f"nexa_O{i}"] = sum(s[name]["result"] in ("T", "t") for s in sheet) / len(sheet)

print(f"{'':<36}" + "".join(f"{n:>10}" for n in cols))
for key, label in METRIC_NAMES.items():
    print(f"  {label:<34}" + "".join(f"{cols[n][key]:>10}" if key == "n" else f"{cols[n][key]:>10.2%}" for n in cols))
print(f"  {'nexa 판정 (논문 표 1)':<34}" + "".join(f"{nexa_tf[n]:>10.2%}" if n in nexa_tf else f"{'':>10}" for n in cols))

                                          ours    public   nexa_O0   nexa_O1   nexa_O2   nexa_O3
  개수                                       210       210       210       210       210       210
  형식 OK                                 98.57%    97.14%    97.14%    97.62%    96.67%    96.67%
  함수 선택 정확도                             89.52%    97.14%    97.14%    96.67%    96.67%    96.67%
  인자 정확도 (함수 맞힌 positive 중)             75.84%    95.36%    95.36%    95.34%    91.71%    91.71%
  전체 정확도 ★                              69.05%    92.86%    92.86%    92.38%    89.05%    89.05%
    positive                            67.50%    92.50%    92.50%    92.00%    88.50%    88.50%
    negative                           100.00%   100.00%   100.00%   100.00%   100.00%   100.00%
  느슨한 정확도 (논문에 없는 추가)                   73.81%    92.86%    92.86%    93.81%    89.52%    89.52%
  호출 문자열 완전 일치                          69.05%    92.86%    92.86%    91.90%    89.05%    89.05%
  nexa 판정 (논문 표 1)            

### 우리 모델과 공개 모델의 결과가 다른 질의
한쪽만 맞힌 줄. 랩탑에서 자세히 보려면 `{EVAL}/{ours,public}/errors.csv`.

In [10]:
sc = {n: load_jsonl(Path(f"{EVAL}/{n}/scored.jsonl")) for n in ("ours", "public")}
for a, b in zip(sc["ours"], sc["public"]):
    if (a["outcome"] == "correct") != (b["outcome"] == "correct"):
        print(f"[{a['index']}] {a['query'][:90]}")
        print(f"   정답   {a['call']}")
        print(f"   ours   {a['outcome']:<26} {a['pred_call']}")
        print(f"   public {b['outcome']:<26} {b['pred_call']}")

[10] Remind me to check on the cake in the oven at 17:15. Please set it with the label 'Cake Ch
   정답   <nexa_10>('17:15', 'Cake Check')<nexa_end>
   ours   function:missed_call       <nexa_20>()<nexa_end>
   public correct                    <nexa_10>('17:15', 'Cake Check')<nexa_end>
[27] I need the lowest brightness setting to conserve energy while my device charges.
   정답   <nexa_11>(0)<nexa_end>
   ours   args:valid_but_different   <nexa_11>(1)<nexa_end>
   public correct                    <nexa_11>(0)<nexa_end>
[30] Set a reminder for 'Mom Birthday Party' on '2024-01-10-18-00' to '2024-01-10-21-00'.
   정답   <nexa_12>('Mom Birthday Party', '2024-01-10-18-00', '2024-01-10-21-00')<nexa_end>
   ours   args:count_mismatch        <nexa_12>('2024-01-10-18-00', '2024-01-10-21-00')<nexa_end>
   public correct                    <nexa_12>('Mom Birthday Party', '2024-01-10-18-00', '2024-01-10-21-00')<nexa_end>
[31] Organize a 'Community Clean-up' starting at '2023-11-15-09-00' and ending at